# PCRGlobWB

Let us look at the Speyside region again!
PCRGlobWB uses a clonemap, this means we will need to make one for the Speyside!
We will do that after the forcing generation.

Please note that when you do this yourself, think of a smart way to divide this notebook into several ones.

In [1]:
# This is only used to suppress some distracting output messages
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import subprocess
import matplotlib.pyplot as plt
from cartopy import crs
from cartopy import feature as cfeature
from rich import print
import pandas as pd
import xarray as xr
from pathlib import Path
from datetime import datetime
from ipywidgets import IntProgress
from IPython.display import display
import fiona
import shapely.geometry
from pyproj import Geod
import logging

import ewatercycle.forcing
import ewatercycle.models
import ewatercycle.parameter_sets

logger = logging.getLogger("esmvalcore")
logger.setLevel(logging.WARNING)

In [2]:
# Region selection using Caravan
"""Avon at Delnashaugh, in Scotland"""
caravan_id = "camelsgb_8004"

# Time periods for experiment, note they are hydrological years
experiment_start_date = "2005-08-01T00:00:00Z"
experiment_end_date = "2008-08-31T00:00:00Z"

In [3]:
forcing_path_caravan = Path.home() / "forcing" / caravan_id / "caravan"
forcing_path_caravan.mkdir(exist_ok=True, parents=True)

forcing_path_pcrglobwb = Path.home() / "forcing" / caravan_id / "pcrglobwb"
forcing_path_pcrglobwb.mkdir(exist_ok=True)

prepared_pcrglobwb_forcing = forcing_path_pcrglobwb / "work/diagnostic/script"

pcr_glob_directory = Path("/data/shared/parameter-sets/pcrglobwb_global")

In [5]:
try:
    # Option two: load data that you or someone else generated previously
    caravan_forcing = ewatercycle.forcing.sources['CaravanForcing'].load(directory=forcing_path_caravan)
except:
    # Option one: generate forcing data
    caravan_forcing = ewatercycle.forcing.sources['CaravanForcing'].generate(
        start_time=experiment_start_date,
        end_time=experiment_end_date,
        directory=forcing_path_caravan,
        basin_id=caravan_id,
    )


RuntimeError: Failed to decode variable 'time': NetCDF: DAP failure

Now we will generate the PCRGlobWB forcing.

In [ ]:
# Set the extent of the forcing data sufficiently larger than the clonemap.
esmvaltool_padding = 1.5

region_Scotland = {
        "start_longitude": -9-esmvaltool_padding,
        "end_longitude": -1+esmvaltool_padding,
        "start_latitude": 50-esmvaltool_padding,
        "end_latitude": 61+esmvaltool_padding,
}

climate_start_date = experiment_start_date
years_of_climatology = 1
climate_end_date = f"{int(experiment_start_date[:4]) + years_of_climatology}{experiment_start_date[4:]}"

pcrglobwb_forcing = ewatercycle.forcing.sources["PCRGlobWBForcing"].generate(
    dataset="ERA5",
    start_time=experiment_start_date,
    end_time=experiment_end_date,
    start_time_climatology=experiment_start_date,
    end_time_climatology=climate_end_date,  # one year more than the start date
    shape=caravan_forcing.shape,
    extract_region=region_Scotland,
    directory = forcing_path_pcrglobwb
)


pcrglobwb_forcing = ewatercycle.forcing.sources["PCRGlobWBForcing"].load(
    directory=prepared_pcrglobwb_forcing,
)

print(pcrglobwb_forcing)

In [ ]:
print(pcrglobwb_forcing)

## Plot the forcing

In [ ]:
for file_name in [pcrglobwb_forcing.temperatureNC, pcrglobwb_forcing.precipitationNC]:
    dataset = xr.load_dataset(f"{pcrglobwb_forcing.directory}/{file_name}")
    # print(dataset)
    # print("------------------------")
    var = list(dataset.data_vars.keys())[0]
    dataset[var].isel(time=-1).plot(cmap="coolwarm", robust=True, size=5)

## The clonemap

In [ ]:
def create_clonemap(lonmin, latmin, lonmax, latmax, forcing_resolution, catchment):
    """Create new clonemap compatible with forcing data resolution."""
    dlon = lonmax - lonmin
    dlat = latmax - latmin

    msg = (
        "The clonemap extent divided by the forcing resolution must yield"
        "an integer number of grid cells."
    )
    assert dlon % forcing_resolution == 0, f"Longitudes not compatible. {msg}"
    assert dlat % forcing_resolution == 0, f"Latitudes not compatible. {msg}"

    clonemap_dir = (
        "/data/shared/parameter-sets/pcrglobwb_global/global_05min/cloneMaps"
    )
    globalclonemap = clonemap_dir + "/clone_global_05min.map"
    outputclonemap = forcing_path_pcrglobwb / f"{catchment.lower()}_05min.map"  # copy to clonemap dir after ensuring it is correct

    subprocess.call(
        f"gdal_translate -of PCRaster {globalclonemap} -projwin "
        f"{lonmin} {latmax} {lonmax} {latmin} {outputclonemap}",
        shell=True,
    )
    return outputclonemap

In [ ]:
# Catchment bounding boxes
clonemap_extents = {
    "Speyside": {"latitude": (57, 58), "longitude": (-4.5, -2.5)},
}

forcing_resolution = 0.75
for (catchment, extents) in clonemap_extents.items():
    latmin, latmax = extents["latitude"]
    lonmin, lonmax = extents["longitude"]
    print(
        create_clonemap(lonmin, latmin, lonmax, latmax, forcing_resolution, catchment)
    )

## Parameter set

In [ ]:
# Station BOAT O BRIG, river Spey
station_latitude = 	57.55104
station_longitude = -3.14039

In [ ]:
# Parameter set
parameter_set = ewatercycle.parameter_sets.ParameterSet(
    name="custom_parameter_set",
    directory=pcr_glob_directory,
    config=forcing_path_pcrglobwb / f"speyside_05min.map",
    target_model="pcrglobwb",
    supported_model_versions={"setters"},
)

In [ ]:
print(parameter_set)

## PCRGlobWB Model

In [ ]:
pcrglob = ewatercycle.models.PCRGlobWB(
    parameter_set=parameter_set,
    forcing=pcrglobwb_forcing
)

print(pcrglob)

In [ ]:
print(pcrglob.parameters)

In [ ]:
cfg_file, cfg_dir = pcrglob.setup(
    end_time=experiment_end_date,
    max_spinups_in_years=0
)

In [ ]:
# Convert ISO 8601 strings to datetime objects
start_time = datetime.strptime(experiment_start_date, '%Y-%m-%dT%H:%M:%SZ')
end_time = datetime.strptime(experiment_end_date, '%Y-%m-%dT%H:%M:%SZ')

# Calculate the number of days for the progression bar
delta = end_time - start_time
number_of_days = delta.days
print(f"Number of days to model: {number_of_days}")

In [ ]:
# Initialize
pcrglob.initialize(cfg_file)

In [ ]:
time = pd.date_range(pcrglob.start_time_as_isostr, pcrglob.end_time_as_isostr)
timeseries = pd.DataFrame(
    index=pd.Index(time, name="time"), columns=["PCRGlobWB: BOAT O BRIG"]
)

In [ ]:
# Progress bar, since this can take a while
f = IntProgress(min=0, max=number_of_days) # instantiate the bar
display(f) # display the bar

while pcrglob.time < pcrglob.end_time:
    pcrglob.update()

    # Track discharge at station location
    discharge_at_station = pcrglob.get_value_at_coords(
        "discharge", lat=[station_latitude], lon=[station_longitude]
    )
    time = pcrglob.time_as_isostr
    timeseries.loc[time, "PCRGlobWB: BOAT O BRIG"] = discharge_at_station[0]

    # Update progress bar
    f.value += 1

print("Model run finished!")

## Interact with the model

**NOTE** we will need to finalize the model later!!

The model has a lot of variables, you can find them in the section about PCRGlobWB in getting started or

In [ ]:
# List the variables
# list(pcrglob.output_var_names)

In [ ]:
# Getting the discharge

da = pcrglob.get_value_as_xarray("discharge")
da.thin(5)  # only show every 5th value in each dim

In [ ]:
fig = plt.figure(dpi=120)
ax = fig.add_subplot(111, projection=crs.PlateCarree())
da.plot(ax=ax, cmap="GnBu")

# Overlay ocean and coastlines
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.RIVERS, color="k")
ax.coastlines()

# Add a red cross marker at the location of the Leven River at Newby Bridge
ax.scatter(station_longitude, station_latitude, s=250, c="r", marker="x", lw=2)

In [ ]:
# Get the discharge data
timeseries.plot()
plt.ylabel("Discharge $[m^3/s]$")

## Finalize the model

In [ ]:
pcrglob.finalize()